# 03 - Modelo SIR para Simulación de Pandemias

En este cuaderno se implementa el modelo epidemiológico SIR (Susceptibles, Infectados, Recuperados) para simular la propagación del COVID-19 y ajustarlo a los picos reales observados en los datos de Johns Hopkins.

In [3]:
import pandas as pd
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
sns.set_style("whitegrid")

## 1. Definición del Modelo SIR

El modelo SIR se basa en un sistema de ecuaciones diferenciales ordinarias (ODEs):
- dS/dt = -beta * S * I / N
- dI/dt = beta * S * I / N - gamma * I
- dR/dt = gamma * I

In [4]:
def deriv(y, t, N, beta, gamma):
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return dSdt, dIdt, dRdt

## 2. Ajuste del Modelo a Datos Reales

Seleccionamos un país (ej. Italia) y una ventana de tiempo específica (primera ola) para ajustar el modelo.

In [5]:
data_path = Path("..") / "data" / "covid_confirmed_long.csv"
df = pd.read_csv(data_path, parse_dates=['Date'])

country = 'Italy'
df_country = df[df['Country/Region'] == country].groupby('Date')['Confirmed'].sum()
df_country = df_country['2020-02-21':'2020-05-30']

y_data = df_country.values
t_data = np.arange(len(y_data))

print(f"Ventana de tiempo para ajuste: {df_country.index.min()} a {df_country.index.max()}")

Ventana de tiempo para ajuste: 2020-02-21 00:00:00 a 2020-05-30 00:00:00


## 3. Optimización de Parámetros

Utilizamos `scipy.optimize.minimize` para encontrar los valores óptimos de beta y gamma.

In [6]:
def objective_func(params, t, y, N):
    beta, gamma = params
    y0 = S0, I0, R0 = N - 1, 1, 0
    ret = odeint(deriv, y0, t, args=(N, beta, gamma))
    I_pred = ret[:, 1]
    return np.sum((I_pred - y)**2)

N = 60000000 # Población aproximada de Italia
res = minimize(objective_func, [0.2, 0.1], args=(t_data, y_data, N), bounds=[(0, 1), (0, 1)])

beta_opt, gamma_opt = res.x
R0 = beta_opt / gamma_opt

print(f"Beta óptimo: {beta_opt:.4f}")
print(f"Gamma óptimo: {gamma_opt:.4f}")
print(f"R0 calculado: {R0:.4f}")

Beta óptimo: 1.0000
Gamma óptimo: 0.8639
R0 calculado: 1.1575
